# Week 2 · Task 3 — Train a classifier

Can we predict whether someone will return to an app the next day? In this exercise, each simulated session has three features: time spent, screens visited, and taps. The label is 1 if the simulated user returned and 0 otherwise.

From the project root, launch JupyterLab with `uv run --locked --with torch jupyter lab`. Open this notebook in that environment and run its cells in order.

## 1. Create examples and split them

`X` is a table with one row per session and three columns. The numbers represent standardized features, so negative values mean below average. `y` has one label per row. A rule with some random noise creates the labels. For example, a session with features `[1, 0, 0]` and zero noise gets a score of 1.5 from the rule, so its label is 1.

Shuffle the row indices once. The model learns from 210 training sessions. We check progress on 45 separate validation sessions. We reserve 45 test sessions for the final model after comparing settings in Task 4.

In [ ]:
import torch
from torch import nn

rng = torch.Generator().manual_seed(42)

X = torch.randn(300, 3, generator=rng)
noise = 0.3 * torch.randn(300, generator=rng)
y = (1.5 * X[:, 0] + X[:, 1] - 0.5 * X[:, 2] + noise > 0).float()

train, val, test = torch.randperm(len(X), generator=rng).split([210, 45, 45])

print("Features:", X.shape, "| Labels:", y.shape)
print(f"Train: {len(train)} | Validation: {len(val)} | Test: {len(test)}")

## 2. Teach the model with the training sessions

`nn.Linear(3, 1)` uses three trainable weights and one bias to produce one raw score per session. The model starts with random parameters and learns from the feature rows and labels.

Each epoch processes the whole training split and takes one optimizer step:

1. Clear the gradients from the previous epoch.
2. Run the forward pass and calculate the loss with `BCEWithLogitsLoss`.
3. Use `loss.backward()` to compute each parameter's gradient.
4. Use `optimizer.step()` to update the parameters. SGD subtracts the learning rate times the gradient, just like the single weight update in Task 2.

We report training and validation loss after the same update, without computing gradients for either measurement.

In [ ]:
torch.manual_seed(42)

model = nn.Linear(3, 1)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

for epoch in range(1, 101):
    optimizer.zero_grad()

    # Remove the final dimension so scores and labels both have shape (210,).
    scores = model(X[train]).squeeze(-1)
    loss = loss_fn(scores, y[train])
    loss.backward()
    optimizer.step()

    if epoch in (1, 25, 50, 100):
        with torch.no_grad():
            train_loss = loss_fn(model(X[train]).squeeze(-1), y[train])
            val_loss = loss_fn(model(X[val]).squeeze(-1), y[val])

        print(
            f"Epoch {epoch:3d} | "
            f"Train loss: {train_loss.item():.3f} | "
            f"Validation loss: {val_loss.item():.3f}"
        )

## 3. Compare with a simple baseline

A raw score of 0 corresponds to a 50% probability after applying the sigmoid. Predict that the user returns when the score is 0 or higher.

The baseline always predicts the most common label in the training split. Score both methods on the same validation sessions, then compare their accuracy.

In [ ]:
with torch.no_grad():
    val_scores = model(X[val]).squeeze(-1)
    predictions = val_scores >= 0
    val_labels = y[val].bool()
    model_accuracy = (predictions == val_labels).float().mean()

    majority_label = y[train].mean() >= 0.5
    baseline_accuracy = (majority_label == val_labels).float().mean()

print(f"Validation accuracy: {model_accuracy.item():.1%}")
print(f"Baseline accuracy:   {baseline_accuracy.item():.1%}")

## What to record

- Compare the training loss at epochs 1 and 100.
- Record the validation accuracy for the classifier and the baseline.
- In Task 4, reuse the same split and the same parameter initialization when you compare learning rates.